# Gravitational Lens Detection using ResNet18 (Baseline)

This notebook implements a deep learning pipeline to classify astronomical images into:
- Lens (1)
- Non-Lens (0)

The goal is to establish a strong baseline model and evaluate its robustness under different perturbations such as noise, blur, and low-light conditions.

# 1. IMPORTS

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

from src.data import LensDataset, build_train_test_samples
from src.model import build_resnet18_binary

# 2. DEVICE

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# 4. LOAD DATA

In [ ]:
data_root = "data"

train_samples, test_samples = build_train_test_samples(data_root)

print("Train:", len(train_samples))
print("Test:", len(test_samples))

# 5. TRAIN/VAL SPLIT (90:10)

In [ ]:
labels = [s.label for s in train_samples]

train_idx, val_idx = train_test_split(
    np.arange(len(train_samples)),
    test_size=0.1,
    stratify=labels,
    random_state=42
)

train_split = [train_samples[i] for i in train_idx]
val_split = [train_samples[i] for i in val_idx]

print("Train:", len(train_split), "Val:", len(val_split))

# 6. TRANSFORMS

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

# 7. DATALOADERS

In [ ]:
train_dataset = LensDataset(train_split, transform=train_transform)
val_dataset = LensDataset(val_split, transform=eval_transform)
test_dataset = LensDataset(test_samples, transform=eval_transform)

# class imbalance handling
labels_tensor = torch.tensor([s.label for s in train_split])
class_counts = torch.bincount(labels_tensor)

weights = 1.0 / class_counts.float()
sample_weights = weights[labels_tensor]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

# 8. MODEL

In [ ]:
model = build_resnet18_binary(pretrained=True).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# 9. TRAIN FUNCTION

In [ ]:
def train_one_epoch():
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

# 10. EVALUATE

In [ ]:
def evaluate(loader):
    model.eval()
    y_true, y_score = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:,1]

            y_true.extend(labels.numpy())
            y_score.extend(probs.cpu().numpy())

    auc = roc_auc_score(y_true, y_score)
    return auc, y_true, y_score

# 11. TRAINING LOOP

In [ ]:
for epoch in range(5):
    loss = train_one_epoch()
    val_auc, _, _ = evaluate(val_loader)

    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Val AUC: {val_auc:.4f}")

# 12. TEST EVALUATION

In [ ]:
test_auc, y_true, y_score = evaluate(test_loader)
print("Test AUC:", test_auc)

# 13. ROC CURVE

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_score)

plt.plot(fpr, tpr)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.show()

# 14. ROBUSTNESS TEST

In [ ]:
def test_robustness(mode):
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor()
    ])

    if mode == "noise":
        transform = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Lambda(lambda x: torch.clamp(x + 0.1*torch.randn_like(x),0,1))
        ])

    dataset = LensDataset(test_samples, transform=transform)
    loader = DataLoader(dataset, batch_size=32)

    auc, _, _ = evaluate(loader)
    print(mode, "AUC:", auc)

In [ ]:
for m in ["clean","noise","blur","low_light"]:
    test_robustness(m)

#  15. CONCLUSION

## Conclusion

- Baseline model achieves high ROC-AUC on clean data
- Performance drops under noise
- Indicates sensitivity to domain shift

This motivates the use of domain adaptation methods such as WDGRL.